# Phase 9 — Quantitative Analysis

Requires `results/eval_all.json` (run `scripts/eval_all.py` first).

Contents:
1. Confusion matrix (A) Baseline vs (B) Ours
2. Class-wise F1 delta bar chart
3. Per-emotion gaze feature box plot

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..')
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
FEAT_NAMES = ['p_face', 'p_out', 'switch_rate', 'entropy', 'target_count', 'p_center']

with open(ROOT / 'results' / 'eval_all.json') as f:
    results = json.load(f)

print('Conditions found:', list(results.keys()))

In [ ]:
# 1. Confusion matrices
def plot_cm(cm, title, ax):
    cm_arr = np.array(cm)
    cm_norm = cm_arr.astype(float) / cm_arr.sum(axis=1, keepdims=True).clip(1)
    sns.heatmap(cm_norm, annot=cm_arr, fmt='d', cmap='Blues',
                xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if 'A_baseline' in results:
    plot_cm(results['A_baseline']['confusion_matrix'], '(A) Baseline', axes[0])
if 'B_ours' in results:
    plot_cm(results['B_ours']['confusion_matrix'], '(B) + Gaze (ours)', axes[1])
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'figures' / 'confusion_matrices.png'), dpi=120)
plt.show()

In [ ]:
# 2. Class-wise F1 delta
if 'A_baseline' in results and 'B_ours' in results:
    cf1_a = results['A_baseline']['class_f1']
    cf1_b = results['B_ours']['class_f1']
    deltas = [cf1_b.get(e,0) - cf1_a.get(e,0) for e in EMOTION_LABELS]
    colors = ['green' if d >= 0 else 'red' for d in deltas]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(EMOTION_LABELS, deltas, color=colors, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Class-wise F1 Delta: (B) − (A)')
    ax.set_ylabel('ΔF1')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'results' / 'figures' / 'classwise_f1_delta.png'), dpi=120)
    plt.show()
    print('Best improved emotion:', EMOTION_LABELS[np.argmax(deltas)])
    print('Worst regression:     ', EMOTION_LABELS[np.argmin(deltas)])

In [ ]:
# 3. Per-emotion gaze feature box plot (train)
feat_path = ROOT / 'features' / 'gaze' / 'train.pkl'
if feat_path.exists():
    import csv
    with feat_path.open('rb') as f:
        feats = pickle.load(f)

    csv_p = ROOT / 'data' / 'MELD.Raw' / 'train_sent_emo.csv'
    if not csv_p.exists():
        csv_p = ROOT / 'data' / 'MELD.Raw' / 'train_meld_emo.csv'
    labels = {}
    with csv_p.open() as f:
        for row in csv.DictReader(f):
            labels[(int(row['Dialogue_ID']),int(row['Utterance_ID']))] = row['Emotion'].lower()

    rows = []
    for (d,u), vec in feats.items():
        emo = labels.get((d,u), 'unknown')
        rows.append({'emotion': emo, **{FEAT_NAMES[i]: float(vec[i]) for i in range(6)}})
    df = pd.DataFrame(rows)
    df = df[df['emotion'].isin(EMOTION_LABELS)]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for i, feat in enumerate(FEAT_NAMES):
        ax = axes[i//3][i%3]
        data = [df.loc[df['emotion']==e, feat].values for e in EMOTION_LABELS]
        ax.boxplot(data, labels=EMOTION_LABELS, showfliers=False)
        ax.set_title(feat)
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('Gaze Features per Emotion (train)')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'results' / 'figures' / 'gaze_per_emotion_boxplot.png'), dpi=120)
    plt.show()
else:
    print('Feature file not found — run build_features.py first.')